# Split Data
Stratified 60 / 20 / 20 split into training, calibration and test sets.

In [1]:
%pip install pandas scikit-learn

In [2]:
import os

import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
processed_data_folder = "../../data/processed"
tables_folder = "../../results/tables"
os.makedirs(tables_folder, exist_ok=True)

clean_data = pd.read_csv(processed_data_folder + "/diabetes_clean.csv")
print(clean_data.shape)

(418492, 19)


## Split

In [4]:
random_seed = 42

# first take 60% for training, then split the other 40% in half
train_data, other_data = train_test_split(
    clean_data,
    test_size=0.4,
    stratify=clean_data["diabetes"],
    random_state=random_seed,
)

calibration_data, test_data = train_test_split(
    other_data,
    test_size=0.5,
    stratify=other_data["diabetes"],
    random_state=random_seed,
)

print("Training:", train_data.shape)
print("Calibration:", calibration_data.shape)
print("Test:", test_data.shape)

Training: (251095, 19)
Calibration: (83698, 19)
Test: (83699, 19)


## Check the class balance in each set

In [5]:
data_sets = {
    "Training": train_data,
    "Calibration": calibration_data,
    "Test": test_data,
}

rows = []
for name, data in data_sets.items():
    rows.append({
        "set": name,
        "records": len(data),
        "percent_of_data": round(len(data) / len(clean_data) * 100, 1),
        "diabetes_cases": data["diabetes"].sum(),
        "diabetes_rate": round(data["diabetes"].mean() * 100, 2),
    })

split_sizes = pd.DataFrame(rows)
split_sizes.to_csv(tables_folder + "/table_split_sizes.csv", index=False)
split_sizes

,set,records,percent_of_data,diabetes_cases,diabetes_rate
0,Training,251095,60.0,35872,14.29
1,Calibration,83698,20.0,11957,14.29
2,Test,83699,20.0,11957,14.29


## Subgroup sizes in the test set

In [6]:
# same age bands as in the EDA notebook
def add_age_band(data):
    data = data.copy()
    data["age_band"] = pd.cut(
        data["age_group"],
        bins=[0, 3, 6, 9, 13],
        labels=["18-34", "35-49", "50-64", "65+"],
    )
    data["sex_label"] = data["sex"].map({1: "Male", 0: "Female"})
    return data


test_with_bands = add_age_band(test_data)

test_subgroups = test_with_bands.groupby(["age_band", "sex_label"], observed=True)["diabetes"].agg(["size", "sum", "mean"])
test_subgroups.columns = ["records", "diabetes_cases", "diabetes_rate"]
test_subgroups["diabetes_rate"] = (test_subgroups["diabetes_rate"] * 100).round(2)

test_subgroups.to_csv(tables_folder + "/table_test_subgroups.csv")
test_subgroups

records  diabetes_cases  diabetes_rate
age_band sex_label                                        
18-34    Female        6512             145           2.23
         Male          7672             137           1.79
35-49    Female        8076             584           7.23
         Male          7722             495           6.41
50-64    Female       11071            1768          15.97
         Male          9751            1801          18.47
65+      Female       17472            3434          19.65
         Male         13940            3394          24.35

In [7]:
# people with an unknown age are left out of the age subgroup results
print("Test records with unknown age:", test_data["age_group"].isna().sum())
print("Smallest number of cases in a subgroup:", test_subgroups["diabetes_cases"].min())

Test records with unknown age: 1483
Smallest number of cases in a subgroup: 137


## Save

In [8]:
train_data.to_csv(processed_data_folder + "/train.csv", index=False)
calibration_data.to_csv(processed_data_folder + "/calibration.csv", index=False)
test_data.to_csv(processed_data_folder + "/test.csv", index=False)

print("Saved training, calibration and test sets")

Saved training, calibration and test sets
